# HDF5 Molecule Dataset Exploration

Inspect and visualize molecular property data stored in HDF5 files produced by the generative-model evaluation pipeline.

**What this notebook does**

1. Inspects the internal tree structure of an HDF5 file (groups / molecules / datasets).
2. Plots the distribution of total atomic forces per group, for one or several models/targets.
3. Lets you interactively browse a single property, a pair of properties, or a triple of properties across all molecules in a file, using dropdown widgets.

**How to use it**
- Fill in the paths in the *Configuration* cell below — this is the only place you need to edit before running the rest of the notebook.
- Run cells top to bottom. The widget-based sections (property distributions) require `ipywidgets` to render; if widgets don't show up, check that the ipywidgets Jupyter extension is enabled for your environment.

**Expected HDF5 layout**
Each file is expected to contain top-level groups (e.g. `G-active-loop-0`, `G-active-loop-1`, ...), each holding one sub-group per molecule, and each molecule holding named property datasets such as `eMBD`, `HLgap`, `atNUM`, `totFOR`, etc.

## Setup: install dependencies
Run once per environment.

In [ ]:
%pip install -q pandas h5py rich matplotlib ipywidgets plotly ipympl tqdm


## Configuration
Set these paths once, then run the rest of the notebook. All other cells reference these variables instead of hardcoded paths.

In [ ]:
from pathlib import Path

# Root directory containing the experiment sub-folders (data_full_dataset, data_embd_jmp, ...).
DATA_ROOT = Path("/path/to/your/data")  # <-- SET THIS

# Where generated plots (.png) get saved.
RESULTS_DIR = Path("/path/to/your/results")  # <-- SET THIS
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Convenience paths to specific experiment folders used throughout this notebook.
DATA_FULL_DATASET = DATA_ROOT / "data_full_dataset"
DATA_EMBD_JMP = DATA_ROOT / "data_embd_jmp"
DATA_EMBD_HLGAP = DATA_ROOT / "data_embd_hlgap"
DATA_TRIANGLE_TRAJ = DATA_ROOT / "data_triangle_traj"

# Individual HDF5 files referenced in the examples below - point these at real files
# once DATA_ROOT is set correctly, or replace with any file you want to explore.
HDF5_EXAMPLE_FILE = DATA_FULL_DATASET / "evaluation/target0/model0/hdf5/top_20-eMBD-m0-t0-extra_properties.hdf5"
HDF5_FULL_DATASET_OUTPUT = DATA_FULL_DATASET / "hdf5/qm7x-aqm-gas-noP-output.hdf5"
HDF5_TRIANGLE_TRAJ_OUTPUT = DATA_TRIANGLE_TRAJ / "hdf5/qm7x-aqm-gas-noP-output.hdf5"
HDF5_EMBD_JMP_MOLECULES = DATA_EMBD_JMP / "hdf5/molecules-fps-40000-output.hdf5"
HDF5_EMBD_HLGAP_MOLECULES = DATA_EMBD_HLGAP / "hdf5/molecules-fps-40000-output.hdf5"


#### Required imports

In [ ]:
# Read an HDF5 dataset
import numpy as np
import pandas as pd
import h5py
from rich.console import Console
from rich.tree import Tree
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
from tqdm.notebook import tqdm


In [ ]:
# Create a console object for rich output, and allow scrollable output cells
console = Console()

display(HTML('''
<style>
.output_scroll {
    display: block;
    max-height: 300px;
    max-width: 100%;
    overflow: auto;
}
</style>
'''))


# Visualize file structure

In [ ]:
def build_tree(group, tree, max_groups=3):
    """
    Recursively build a tree structure from the HDF5 group.

    Parameters
    ----------
    group : h5py.Group
        The HDF5 group to build the tree from.
    tree : rich.tree.Tree
        The tree structure to populate.
    max_groups : int, optional
        The maximum number of top-level groups to display.
    """
    group_count = 0
    for key in group.keys():
        branch = tree.add(f"{key}")
        item = group[key]
        if isinstance(item, h5py.Group):
            if group.name == "/":
                group_count += 1
                if group_count > max_groups:
                    tree.add("...")
                    break
            build_tree(item, branch, max_groups)
        elif isinstance(item, h5py.Dataset):
            dtype = item.dtype
            shape = item.shape
            tree.add(f"[green]{key}[/] [dim](dtype={dtype}, shape={shape})[/]")


In [ ]:
# Display the tree structure of the HDF5 file (limited to 3 top-level groups for readability)
with h5py.File(HDF5_EXAMPLE_FILE, "r") as f:
    tree = Tree(HDF5_EXAMPLE_FILE.name)
    build_tree(f, tree, max_groups=3)
    console.print(tree)

# Print the keys and values for the first molecule in the first group
with h5py.File(HDF5_EXAMPLE_FILE, "r") as f:
    first_group = list(f.keys())[0]
    first_molecule = list(f[first_group].keys())[0]
    print(f"First group: {first_group}")
    print(f"First molecule: {first_molecule}")
    for key in f[first_group][first_molecule].keys():
        value = f[first_group][first_molecule][key][()]
        print(f"{key}: {value}")


In [ ]:
# Distribution of the magnitude of total atomic forces (totFOR), grouped by HDF5 group, as a boxplot
with h5py.File(HDF5_EXAMPLE_FILE, "r") as f:
    group_names = list(f.keys())
    data = {}
    for group in tqdm(group_names, desc="Processing groups"):
        molecule_names = list(f[group].keys())
        forces = []
        for molecule in molecule_names:
            if "totFOR" not in f[group][molecule]:
                print(f"Warning: 'totFOR' dataset not found for molecule '{molecule}' in group '{group}'. Skipping.")
                continue
            atomicFOR = f[group][molecule]["totFOR"][()]
            molFOR = np.sum(abs(atomicFOR), axis=0)
            net_force = np.linalg.norm(molFOR)
            forces.append(net_force)
        data[group] = forces

plt.figure(figsize=(12, 6))
plt.boxplot(data.values(), labels=data.keys())
plt.title("Distribution of Magnitude of Total Forces by Group")
plt.xlabel("Group")
plt.ylabel("Magnitude of Total Forces")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()


Same plot, repeated per target/model combination, using files under `DATA_FULL_DATASET/evaluation/target{t}/model{m}/hdf5/`.

In [ ]:
for t in range(4):
    data = {}
    for m in range(4):
        hdf5_path = DATA_FULL_DATASET / f"evaluation/target{t}/model{m}/hdf5/m{m}-t{t}-extra_properties.hdf5"
        with h5py.File(hdf5_path, "r") as f:
            group_name = str(list(f.keys())[0])
            molecule_names = list(f[group_name].keys())
            forces = []
            for molecule in molecule_names:
                if "totFOR" not in f[group_name][molecule]:
                    print(f"Warning: 'totFOR' dataset not found for molecule '{molecule}' in group '{group_name}' in model {m}. Skipping.")
                    continue
                atomicFOR = f[group_name][molecule]["totFOR"][()]
                molFOR = np.sum(abs(atomicFOR), axis=0)
                net_force = np.linalg.norm(molFOR)
                forces.append(net_force)
        data[f"{group_name}-m{m}-t{t}"] = forces

    fig, ax = plt.subplots(figsize=(12, 6))
    bp = ax.boxplot(
        data.values(),
        labels=data.keys(),
        patch_artist=True,
        medianprops={"color": "#d1495b", "linewidth": 2},
        whiskerprops={"color": "#5c677d"},
        capprops={"color": "#5c677d"},
    )
    for box in bp["boxes"]:
        box.set_facecolor("#89c2d9")
        box.set_alpha(0.75)
        box.set_edgecolor("#3a506b")

    ax.set_title(f"Distribution of Magnitude of Total Forces by model for target {t}")
    ax.set_xlabel("Group")
    ax.set_ylabel("Magnitude of Total Forces")
    ax.tick_params(axis="x", rotation=45, colors="#33415c")
    ax.tick_params(axis="y", colors="#33415c")
    ax.grid(axis="y", linestyle="--", alpha=0.3, color="#adb5bd")
    ax.set_facecolor("#f8f9fa")
    plt.tight_layout()
    plt.show()


Same, but restricted to the top-20 molecules per target/model (`top_20-{prop}-...` files).

In [ ]:
prop = "eMBD"
for t in range(4):
    data = {}
    for m in range(4):
        hdf5_path = DATA_FULL_DATASET / f"evaluation/target{t}/model{m}/hdf5/top_20-{prop}-m{m}-t{t}-extra_properties.hdf5"
        with h5py.File(hdf5_path, "r") as f:
            group_name = str(list(f.keys())[0])
            molecule_names = list(f[group_name].keys())
            forces = []
            for molecule in molecule_names:
                if "totFOR" not in f[group_name][molecule]:
                    print(f"Warning: 'totFOR' dataset not found for molecule '{molecule}' in group '{group_name}' in model {m}. Skipping.")
                    continue
                atomicFOR = f[group_name][molecule]["totFOR"][()]
                molFOR = np.sum(abs(atomicFOR), axis=0)
                net_force = np.linalg.norm(molFOR)
                forces.append(net_force)
        data[f"{group_name}-m{m}-t{t}"] = forces

    fig, ax = plt.subplots(figsize=(12, 6))
    bp = ax.boxplot(
        data.values(),
        labels=data.keys(),
        patch_artist=True,
        medianprops={"color": "#d1495b", "linewidth": 2},
        whiskerprops={"color": "#5c677d"},
        capprops={"color": "#5c677d"},
    )
    for box in bp["boxes"]:
        box.set_facecolor("#89c2d9")
        box.set_alpha(0.75)
        box.set_edgecolor("#3a506b")

    ax.set_title(f"Distribution of Magnitude of Total Forces of top 20 molecules by model for target {t}")
    ax.set_xlabel("Group")
    ax.set_ylabel("Magnitude of Total Forces")
    ax.tick_params(axis="x", rotation=45, colors="#33415c")
    ax.tick_params(axis="y", colors="#33415c")
    ax.grid(axis="y", linestyle="--", alpha=0.3, color="#adb5bd")
    ax.set_facecolor("#f8f9fa")
    plt.tight_layout()
    plt.show()


### Check data
Quick sanity checks on a few specific files.

In [ ]:
# Full-dataset diffusion output
with h5py.File(HDF5_FULL_DATASET_OUTPUT, "r") as f:
    tree = Tree("diffused-qm7x-aqm-gas-noP")
    build_tree(f, tree, max_groups=3)
    console.print(tree)


In [ ]:
# Look for molecules under group "G-active-loop-2" in the triangle-trajectory output
with h5py.File(HDF5_TRIANGLE_TRAJ_OUTPUT, "r") as f:
    grp = f["G-active-loop-2"]
    print(f"Number of molecules in G-active-loop-2: {len(grp)}")
    print(f"molecule IDs: {list(grp.keys())}")

    mol_id = list(grp.keys())[0]
    print(f"First molecule ID: {mol_id}")
    tree = Tree("aqm_qm7x")
    build_tree(grp[mol_id], tree, max_groups=3)
    console.print(tree)

    for key, value in grp[mol_id].items():
        print(f"{key}: {value[()]}")


In [ ]:
# Count molecules per active-learning group in the embd_jmp molecules file
with h5py.File(HDF5_EMBD_JMP_MOLECULES, "r") as f:
    for i in range(8):
        group_name = f"G-active-loop-{i}"
        grp = f[group_name]
        print(f"Number of molecules in {group_name}: {len(grp)}")


# Visualize properties

The next three sections (one/two/three-property distributions) all follow the same pattern: `extract_properties*` pulls the requested properties out of the HDF5 file, then `show_properties*`/`plot_property_dist` renders them (histogram, scatter, or hexbin). If you need a 4th property or a different plot type, copy the closest of the three and adjust it rather than starting from scratch.

## One property distribution

In [ ]:
def plot_property_dist(dataset, prop="atNUM", save=True):
    """
    Plot the distribution of a specified property (default 'atNUM') across all molecules in a dataset.

    Parameters
    ----------
    dataset : str or Path
        Path to the HDF5 dataset.
    prop : str, optional
        The property to plot (default 'atNUM').
    save : bool, optional
        Whether to save the plot to RESULTS_DIR.
    """
    with h5py.File(dataset, "r") as f:
        property_values = []
        for subset in f.keys():
            if isinstance(f[subset], h5py.Group):
                for molecule in f[subset].keys():
                    if prop in f[subset][molecule]:
                        if prop == "atNUM":
                            property_values.append(len(f[subset][molecule][prop][()]))
                        else:
                            property_values.append(f[subset][molecule][prop][()])

    property_values = np.array(property_values).flatten()
    plt.figure(figsize=(10, 6))
    plt.hist(property_values, bins=10, edgecolor="black", alpha=0.7)
    plt.xlabel(prop)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {prop} in Molecules")
    plt.xticks(fontsize=5)
    plt.grid(axis="y", alpha=0.75)
    plt.tight_layout()
    if save:
        plt.savefig(RESULTS_DIR / f"{prop}_distribution.png", dpi=300)
    plt.show()


Dropdown UI to pick a file and property, then plot its distribution.

In [ ]:
molecule_prop = [
    'DIP',
    'HLgap',
    'atNUM',
    'atXYZ',
    'eAT',
    'eC',
    'eEE',
    'eH',
    'eKIN',
    'eL',
    'eMBD',
    'eNE',
    'eNN',
    'ePBE0',
    'ePBE0+MBD',
    'eTS',
    'eX',
    'eXC',
    'eXX',
    'mC6',
    'mPOL',
]

file_dropdown = widgets.Dropdown(
    options={"Full dataset output": HDF5_FULL_DATASET_OUTPUT, "Triangle traj output": HDF5_TRIANGLE_TRAJ_OUTPUT},
    description="File:", style={"description_width": "initial"},
)
prop1_dropdown = widgets.Dropdown(options=molecule_prop, description="Molecule Property:", style={"description_width": "initial"})
show_button = widgets.Button(description="Show Plot")
output1 = widgets.Output()

def on_show_plot(b):
    with output1:
        clear_output()
        plot_property_dist(file_dropdown.value, prop=prop1_dropdown.value)

show_button.on_click(on_show_plot)
display(widgets.VBox([widgets.HBox((file_dropdown, prop1_dropdown, show_button)), output1]))


## Two properties distribution
Scatter two properties against each other, colored by active-learning group.

In [ ]:
def extract_properties2(file_path, property1, property2):
    """
    Extract arrays for two properties from an HDF5 file, tracking which
    G-active-loop group each point comes from.

    Returns
    -------
    tuple
        (property1_values, property2_values, group_labels, group_numbers)
    """
    property1_values, property2_values = [], []
    group_labels, group_numbers = [], []

    with h5py.File(file_path, "r") as f:
        for subset in f.keys():
            if not isinstance(f[subset], h5py.Group) or "G-active-loop" not in subset:
                continue
            try:
                group_num = int(subset.split("-")[-1])
            except (ValueError, IndexError):
                continue

            for geom in f[subset].keys():
                if property1 in f[subset][geom].keys() and property2 in f[subset][geom].keys():
                    prop1 = f[subset][geom][property1][()]
                    prop2 = f[subset][geom][property2][()]
                    if hasattr(prop1, "shape") and prop1.shape == ():
                        prop1 = float(prop1)
                    if hasattr(prop2, "shape") and prop2.shape == ():
                        prop2 = float(prop2)
                    property1_values.append(prop1)
                    property2_values.append(prop2)
                    group_labels.append(subset)
                    group_numbers.append(group_num)

    return property1_values, property2_values, group_labels, group_numbers


In [ ]:
def show_properties2(prop1_name, prop1_values, prop2_name, prop2_values, group_labels=None, group_numbers=None, save=True):
    """Scatter-plot two properties, colored by G-active-loop group when available."""
    if prop1_name == "atNUM":
        prop1_values = [len(v) for v in prop1_values]
    if prop2_name == "atNUM":
        prop2_values = [len(v) for v in prop2_values]

    prop1_values = np.array(prop1_values).flatten()
    prop2_values = np.array(prop2_values).flatten()

    plt.figure(figsize=(10, 7))

    if group_numbers is not None and len(group_numbers) > 0:
        unique_groups = sorted(set(group_numbers))
        colors = plt.cm.tab10(np.linspace(0, 1, len(unique_groups)))
        for idx, group_num in enumerate(unique_groups):
            mask = np.array(group_numbers) == group_num
            plt.scatter(prop1_values[mask], prop2_values[mask],
                        s=10, alpha=0.6, label=f"G-active-loop-{group_num}", color=colors[idx])
        plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
    else:
        plt.scatter(prop1_values, prop2_values, s=10, alpha=0.5)

    plt.xlabel(prop1_name, fontsize=12)
    plt.ylabel(prop2_name, fontsize=12)
    plt.title(f"Plot of {prop1_name} vs {prop2_name}", fontsize=14)
    plt.margins(x=0.2, y=0.2)
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    if save:
        plt.savefig(RESULTS_DIR / f"plot_{prop1_name}_vs_{prop2_name}_by_group.png", dpi=300)
    plt.show()


Dropdown UI comparing two files (set `file_path1` / `file_path2` in the Configuration cell) across two chosen properties.

In [ ]:
file_path1 = HDF5_FULL_DATASET_OUTPUT
file_path2 = HDF5_EMBD_HLGAP_MOLECULES

prop1_dropdown = widgets.Dropdown(options=molecule_prop, description="Property 1:", style={"description_width": "initial"})
prop2_dropdown = widgets.Dropdown(options=molecule_prop, description="Property 2:", style={"description_width": "initial"})
show_button = widgets.Button(description="Show Plot")
output2 = widgets.Output()

def on_show_plot(b):
    with output2:
        clear_output()
        x1, y1, labels1, nums1 = extract_properties2(file_path1, prop1_dropdown.value, prop2_dropdown.value)
        x2, y2, labels2, nums2 = extract_properties2(file_path2, prop1_dropdown.value, prop2_dropdown.value)
        show_properties2(prop1_dropdown.value, x1, prop2_dropdown.value, y1, labels1, nums1)
        show_properties2(prop1_dropdown.value, x2, prop2_dropdown.value, y2, labels2, nums2)

show_button.on_click(on_show_plot)
display(widgets.VBox([prop1_dropdown, widgets.HBox((prop2_dropdown, show_button)), output2]))


## Three properties distribution
Hexbin plot of two properties, colored by the mean of a third property.

In [ ]:
def extract_properties3(file_path, property1, property2, property3):
    """Extract arrays for three properties from the specified HDF5 file."""
    property1_values, property2_values, property3_values = [], [], []

    with h5py.File(file_path, "r") as f:
        for subset in f.keys():
            if isinstance(f[subset], h5py.Group):
                for geom in f[subset].keys():
                    keys = f[subset][geom].keys()
                    if property1 in keys and property2 in keys and property3 in keys:
                        prop1 = f[subset][geom][property1][()]
                        prop2 = f[subset][geom][property2][()]
                        prop3 = f[subset][geom][property3][()]
                        if hasattr(prop1, "shape") and prop1.shape == ():
                            prop1 = float(prop1)
                        if hasattr(prop2, "shape") and prop2.shape == ():
                            prop2 = float(prop2)
                        if hasattr(prop3, "shape") and prop3.shape == ():
                            prop3 = float(prop3)
                        property1_values.append(prop1)
                        property2_values.append(prop2)
                        property3_values.append(prop3)

    return property1_values, property2_values, property3_values


In [ ]:
def show_properties3(prop1_name, prop1_values, prop2_name, prop2_values, prop3_name, prop3_values, save=True):
    """Hexbin plot of prop1 vs prop2, colored by the mean of prop3."""
    if prop1_name == "atNUM":
        prop1_values = [len(v) for v in prop1_values]
    if prop2_name == "atNUM":
        prop2_values = [len(v) for v in prop2_values]
    if prop3_name == "atNUM":
        prop3_values = [len(v) for v in prop3_values]

    prop1_values = np.array(prop1_values).flatten()
    prop2_values = np.array(prop2_values).flatten()
    prop3_values = np.array(prop3_values).flatten()

    plt.figure(figsize=(8, 6))
    plt.hexbin(prop1_values, prop2_values, C=prop3_values, gridsize=100, alpha=1,
               mincnt=1, reduce_C_function=np.mean, cmap="viridis", vmin=5, vmax=100)
    plt.colorbar(label=f"mean {prop3_name}")
    plt.xlabel(prop1_name)
    plt.ylabel(prop2_name)
    plt.title(f"Plot of {prop1_name} vs {prop2_name}")
    plt.margins(x=0.2, y=0.2)
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.tight_layout()
    if save:
        plt.savefig(RESULTS_DIR / f"plot500_{prop1_name}_vs_{prop2_name}_at_{prop3_name}.png", dpi=300)
    plt.show()


Dropdown UI picking three properties from a single file (`file_path`, set below).

In [ ]:
file_path = HDF5_FULL_DATASET_OUTPUT

prop1_dropdown = widgets.Dropdown(options=molecule_prop, description="Property 1:", style={"description_width": "initial"})
prop2_dropdown = widgets.Dropdown(options=molecule_prop, description="Property 2:", style={"description_width": "initial"})
prop3_dropdown = widgets.Dropdown(options=molecule_prop, description="Property 3:", style={"description_width": "initial"})
show_button = widgets.Button(description="Show Plot")
output3 = widgets.Output()

def on_show_plot(b):
    with output3:
        clear_output()
        x, y, z = extract_properties3(file_path, prop1_dropdown.value, prop2_dropdown.value, prop3_dropdown.value)
        show_properties3(prop1_dropdown.value, x, prop2_dropdown.value, y, prop3_dropdown.value, z)

show_button.on_click(on_show_plot)
display(widgets.VBox([prop1_dropdown, widgets.HBox((prop2_dropdown, show_button)), prop3_dropdown, output3]))


# Visualize molecules
*(Placeholder section — add 3D/structure visualization of individual molecules here if needed.)*